In [1]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import polars as pl

from scipy.stats import norm
from sklearn.linear_model import LinearRegression

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rest_of_season.config import (
    SEASON,
    CURRENT_WEEK,
    N_SIMULATIONS,
    RANDOM_SEED,
    CURRENT_OUTPUT_DIR
)

TARGET_WEEK = CURRENT_WEEK

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

ROS_OUTPUT_DIR = CURRENT_OUTPUT_DIR
ROS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Season:", SEASON)
print("Projection week:", TARGET_WEEK)
print("ROS output:", ROS_OUTPUT_DIR)


Season: 2026
Projection week: 11
Simulations: 50,000
Output directory: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_11
Season: 2026
Projection week: 11
ROS output: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_11


In [2]:
remaining_games = pd.read_parquet(
    ROS_OUTPUT_DIR / "remaining_schedule.parquet"
)

team_snapshot = pd.read_parquet(
    ROS_OUTPUT_DIR / "team_snapshot.parquet"
)

scoring_ratings = pd.read_parquet(
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_ros_scoring_ratings.parquet"
)

injury_strength = pd.read_parquet(
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

print("Remaining games:", len(remaining_games))
print("Teams:", len(team_snapshot))
print("Scoring ratings:", len(scoring_ratings))

assert len(team_snapshot) == 32
assert len(scoring_ratings) == 32
assert remaining_games["game_id"].is_unique

Remaining games: 122
Teams: 32
Scoring ratings: 32


In [3]:
strength_lookup = injury_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjusted_team_strength"
    ]
].copy()

assert strength_lookup["team"].is_unique
assert len(strength_lookup) == 32

home_strength = strength_lookup.rename(columns={
    "team": "home_team",
    "weekly_team_strength": "home_base_strength",
    "injury_adjusted_team_strength": "home_injury_strength"
})

away_strength = strength_lookup.rename(columns={
    "team": "away_team",
    "weekly_team_strength": "away_base_strength",
    "injury_adjusted_team_strength": "away_injury_strength"
})

games = (
    remaining_games
    .merge(
        home_strength,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_strength,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)


# Carry forward the current injury-adjusted ratings
# throughout the remaining regular season.

games["home_model_strength"] = games["home_injury_strength"]

games["away_model_strength"] = games["away_injury_strength"]


games["neutral_strength_diff"] = (
    games["home_model_strength"]
    - games["away_model_strength"]
)

assert games[
    ["home_model_strength", "away_model_strength"]
].notna().all().all()

In [4]:
schedule_clean = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

historical_home = schedule_clean.filter(
    (pl.col("season") >= 2015)
    & (pl.col("season") <= 2025)
    & (pl.col("location") == "Home")
    & pl.col("result").is_not_null()
)

HOME_FIELD_ADVANTAGE = (
    historical_home
    .select(pl.col("result").mean())
    .item()
)

historical_rest = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select(["result", "home_rest", "away_rest", "location"])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

rest_model = LinearRegression()

rest_model.fit(
    historical_rest[["home_field", "rest_diff"]],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

historical_margin_std = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(pl.col("result").std())
    .item()
)

print("Home-field advantage:", round(HOME_FIELD_ADVANTAGE, 3))
print("Rest adjustment per day:", round(REST_POINT_VALUE, 3))
print("Historical margin SD:", round(historical_margin_std, 3))

Home-field advantage: 1.764
Rest adjustment per day: 0.181
Historical margin SD: 14.201


In [5]:
required_schedule_cols = [
    "location", "home_rest", "away_rest"
]

missing = [
    c for c in required_schedule_cols
    if c not in games.columns
]

if missing:
    raise KeyError(
        f"Remaining schedule missing columns: {missing}"
    )

if games[["home_rest", "away_rest"]].isna().any().any():
    raise ValueError(
        "Missing future rest-day values. "
        "We need to derive them from the schedule "
        "before generating projections."
    )

games["home_field_adjustment"] = np.where(
    games["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

games["rest_diff"] = (
    games["home_rest"] - games["away_rest"]
).clip(-7, 7)

games["rest_adjustment"] = (
    games["rest_diff"] * REST_POINT_VALUE
)

games["expected_home_margin"] = (
    games["neutral_strength_diff"]
    + games["home_field_adjustment"]
    + games["rest_adjustment"]
)

games["home_win_probability"] = norm.cdf(
    games["expected_home_margin"]
    / historical_margin_std
)

games["away_win_probability"] = (
    1 - games["home_win_probability"]
)

games["predicted_winner"] = np.where(
    games["home_win_probability"] >= 0.5,
    games["home_team"],
    games["away_team"]
)

games["predicted_win_probability"] = np.maximum(
    games["home_win_probability"],
    games["away_win_probability"]
)

In [6]:
home_scoring = scoring_ratings.rename(columns={
    "team": "home_team",
    "current_off_ppg": "home_current_off_ppg",
    "current_def_ppg": "home_current_def_ppg"
})

away_scoring = scoring_ratings.rename(columns={
    "team": "away_team",
    "current_off_ppg": "away_current_off_ppg",
    "current_def_ppg": "away_current_def_ppg"
})

games = (
    games
    .merge(
        home_scoring,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_scoring,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

scoring_cols = [
    "home_current_off_ppg",
    "home_current_def_ppg",
    "away_current_off_ppg",
    "away_current_def_ppg"
]

assert games[scoring_cols].notna().all().all()

games["raw_home_score"] = (
    games["home_current_off_ppg"]
    + games["away_current_def_ppg"]
) / 2

games["raw_away_score"] = (
    games["away_current_off_ppg"]
    + games["home_current_def_ppg"]
) / 2

games["projected_total"] = (
    games["raw_home_score"]
    + games["raw_away_score"]
)

games["projected_home_score"] = (
    games["projected_total"]
    + games["expected_home_margin"]
) / 2

games["projected_away_score"] = (
    games["projected_total"]
    - games["expected_home_margin"]
) / 2

In [7]:

# Diagnose remaining-game counts by week

print("Target week:", TARGET_WEEK)
print("Total remaining games:", len(games))
print("Week column dtype:", games["week"].dtype)

print("\nGames by week:")
display(
    games.groupby("week")
    .size()
    .reset_index(name="game_count")
    .sort_values("week")
)

print("\nWeek 9 games:", len(games.loc[games["week"] == 9]))
print("Weeks 10-18:", len(games.loc[games["week"] > 9]))

print("\nWeek 9 matchups:")
display(
    games.loc[
        games["week"] == 9,
        ["week", "away_team", "home_team"]
    ]
)


Target week: 11
Total remaining games: 122
Week column dtype: int32

Games by week:


,week,game_count
0,11,13
1,12,16
2,13,14
3,14,15
4,15,16
5,16,16
6,17,16
7,18,16



Week 9 games: 0
Weeks 10-18: 122

Week 9 matchups:


,week,away_team,home_team


In [8]:

# weekly_projection_path = (
#     WEEKLY_DATA_DIR
#     / f"week_{TARGET_WEEK:02d}_projections.parquet"
# )

# weekly_games = pd.read_parquet(weekly_projection_path)

# weekly_games = weekly_games.loc[
#     weekly_games["week"] == TARGET_WEEK
# ].copy()

# comparison = games.loc[
#     games["week"] == TARGET_WEEK,
#     [
#         "game_id",
#         "expected_home_margin",
#         "projected_total",
#         "home_win_probability"
#     ]
# ].merge(
#     weekly_games[
#         [
#             "game_id",
#             "expected_home_margin",
#             "projected_total",
#             "home_win_probability"
#         ]
#     ],
#     on="game_id",
#     suffixes=("_ros", "_weekly"),
#     validate="one_to_one"
# )

# assert len(comparison) == len(
#     games.loc[games["week"] == TARGET_WEEK]
# )

# for metric in [
#     "expected_home_margin",
#     "projected_total",
#     "home_win_probability"
# ]:
#     assert np.allclose(
#         comparison[f"{metric}_ros"],
#         comparison[f"{metric}_weekly"],
#         atol=1e-8
#     ), f"Mismatch in {metric}"

# print(
#     f"Week {TARGET_WEEK} projections match "
#     "the weekly model."
# )#


In [9]:
games["matchup"] = (
    games["away_team"] + " @ " + games["home_team"]
)

games["snapshot_week"] = TARGET_WEEK
games["season"] = SEASON

output_columns = [
    "season",
    "snapshot_week",
    "week",
    "game_id",
    "gameday",
    "away_team",
    "home_team",
    "matchup",
    "projected_away_score",
    "projected_home_score",
    "projected_total",
    "expected_home_margin",
    "away_win_probability",
    "home_win_probability",
    "predicted_winner",
    "predicted_win_probability"
]

ros_projections = games[
    output_columns
].sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

assert ros_projections["game_id"].is_unique
assert ros_projections["week"].ge(TARGET_WEEK).all()
assert ros_projections["week"].le(18).all()

ros_projections.to_parquet(
    ROS_OUTPUT_DIR / "ros_game_projections.parquet",
    index=False
)

ros_projections.to_csv(
    ROS_OUTPUT_DIR / "ros_game_projections.csv",
    index=False
)

print("Saved:", len(ros_projections), "remaining games")

display(ros_projections.head(20).round(3))

Saved: 122 remaining games


,season,snapshot_week,week,game_id,gameday,away_team,home_team,matchup,projected_away_score,projected_home_score,projected_total,expected_home_margin,away_win_probability,home_win_probability,predicted_winner,predicted_win_probability
0,2026,11,11,2026_11_IND_HOU,2026-11-19,IND,HOU,IND @ HOU,20.625,27.267,47.892,6.643,0.320,0.680,HOU,0.680
1,2026,11,11,2026_11_ARI_KC,2026-11-22,ARI,KC,ARI @ KC,14.935,30.392,45.327,15.456,0.138,0.862,KC,0.862
2,2026,11,11,2026_11_BAL_CAR,2026-11-22,BAL,CAR,BAL @ CAR,21.343,26.336,47.680,4.993,0.363,0.637,CAR,0.637
3,2026,11,11,2026_11_JAX_NYG,2026-11-22,JAX,NYG,JAX @ NYG,25.671,19.615,45.287,-6.056,0.665,0.335,JAX,0.665
4,2026,11,11,2026_11_LV_DEN,2026-11-22,LV,DEN,LV @ DEN,18.032,29.038,47.070,11.006,0.219,0.781,DEN,0.781
5,2026,11,11,2026_11_MIA_BUF,2026-11-22,MIA,BUF,MIA @ BUF,15.067,32.370,47.437,17.303,0.112,0.888,BUF,0.888
6,2026,11,11,2026_11_MIN_SF,2026-11-22,MIN,SF,MIN @ SF,21.037,22.733,43.770,1.695,0.452,0.548,SF,0.548
7,2026,11,11,2026_11_NO_CHI,2026-11-22,NO,CHI,NO @ CHI,19.051,28.286,47.337,9.235,0.258,0.742,CHI,0.742
8,2026,11,11,2026_11_NYJ_LAC,2026-11-22,NYJ,LAC,NYJ @ LAC,20.058,24.090,44.148,4.032,0.388,0.612,LAC,0.612
9,2026,11,11,2026_11_PIT_PHI,2026-11-22,PIT,PHI,PIT @ PHI,23.833,18.931,42.764,-4.902,0.635,0.365,PIT,0.635
